# Лекция: Дискриминантный анализ в Python

**Дисциплина:** Введение в анализ больших данных

**LDA (линейный дискриминантный анализ)** — классификация с учителем: по признакам отнести объект к одному из **заранее известных** классов.

Отличие от кластеризации: метки классов даны при обучении.

Инструменты: `sklearn.discriminant_analysis.LinearDiscriminantAnalysis`, PCA, MANOVA.

Демо: **penguins** (вид по морфометрии). Примеры **не из лабораторного задания**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import classification_report, accuracy_score
from statsmodels.multivariate.manova import MANOVA

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Данные


In [ ]:
peng = sns.load_dataset("penguins").dropna()
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = peng[features]
y = peng["species"]
print(y.value_counts())
print(X.describe().round(1))


In [ ]:
sns.pairplot(peng, vars=features, hue="species", corner=True, height=1.8)
plt.suptitle("Penguins: признаки по видам", y=1.02)
plt.show()


---
## 2. PCA (разведка)

PCA ищет направления **максимальной дисперсии** (не обязательно разделения классов).


In [ ]:
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

pca = PCA()
scores = pca.fit_transform(Xs)
print("Доля дисперсии:", np.round(pca.explained_variance_ratio_, 3))
print("Накопленная:   ", np.round(np.cumsum(pca.explained_variance_ratio_), 3))
print("\nLoadings:")
print(pd.DataFrame(pca.components_.T, index=features,
                   columns=[f"PC{i+1}" for i in range(4)]).round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for sp in y.unique():
    m = y == sp
    ax.scatter(scores[m, 0], scores[m, 1], label=sp, s=40, alpha=0.75)
load = pca.components_.T * np.sqrt(pca.explained_variance_)
for i, name in enumerate(features):
    ax.arrow(0, 0, load[i, 0]*2, load[i, 1]*2, color="k", alpha=0.6, head_width=0.08)
    ax.text(load[i, 0]*2.2, load[i, 1]*2.2, name, fontsize=8)
ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)")
ax.legend(); ax.set_title("PCA biplot")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
plt.tight_layout(); plt.show()


---
## 3. LDA: обучение и прогноз

Train / test → `LDA().fit` → `predict` → confusion matrix / accuracy.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=7, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

lda = LDA()
lda.fit(X_train, y_train)
y_pred = lda.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nConfusion matrix:")
print(pd.crosstab(y_pred, y_test, rownames=["Predicted"], colnames=["Actual"]))
print("\n", classification_report(y_test, y_pred, digits=3))


In [ ]:
print("Классы:", lda.classes_)
print("Приоры:", np.round(lda.priors_, 3))
print("\nСредние по классам:")
print(pd.DataFrame(lda.means_, index=lda.classes_, columns=features).round(1))
print("\nКоэффициенты LD:")
print(pd.DataFrame(
    lda.scalings_, index=features,
    columns=[f"LD{i+1}" for i in range(lda.scalings_.shape[1])]
).round(3))


### Проекция на LD1–LD2


In [ ]:
X_ld = lda.transform(X)
fig, ax = plt.subplots(figsize=(8, 6))
for sp in y.unique():
    m = y == sp
    ax.scatter(X_ld[m, 0], X_ld[m, 1], label=sp, s=40, alpha=0.8)
ax.set_xlabel("LD1"); ax.set_ylabel("LD2")
ax.legend(); ax.set_title("LDA: LD1–LD2")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
plt.tight_layout(); plt.show()


### Разделимость классов: MANOVA (Wilks)

H0: векторы средних по классам совпадают.


In [ ]:
df_m = X_test.copy()
df_m["cls"] = y_pred.astype(str)
formula = " + ".join(features) + " ~ cls"
print(MANOVA.from_formula(formula, data=df_m).mv_test())


### Как описать результаты

1. Accuracy и confusion matrix на **test**.  
2. Какие классы путаются чаще.  
3. MANOVA: p < 0.05 → классы различаются в пространстве признаков.  
4. Интерпретация коэффициентов LD (scalings).

---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| LDA | `LDA().fit(X_train, y_train)` |
| Класс | `lda.predict(X)` |
| Вероятности | `lda.predict_proba(X)` |
| Координаты LD | `lda.transform(X)` |
| Confusion | `pd.crosstab` / metrics |
| PCA | `PCA().fit_transform(StandardScaler().fit_transform(X))` |
| MANOVA | `MANOVA.from_formula("y1+y2 ~ class", data=df)` |

---
## Что сделать после лекции

1. Повторите PCA + LDA на **других** признаках.  
2. Откройте лабораторное задание и выполните LDA **самостоятельно**.  
3. Помните: PCA максимизирует дисперсию, LDA — **разделимость классов**.

Удачи!
